# BBC Business → Article Links → CSV (Requests + BeautifulSoup)

This notebook fetches the BBC Business landing page, extracts article links, then visits each article to collect title, published time, and summary, and saves the output to `bbc_business_articles.csv`.

## Install
Run once:
```bash
pip install requests beautifulsoup4 lxml pandas
```

## Notes
- This approach avoids Selenium.
- If you get zero links, BBC may be serving a consent/variant page. In that case, open the printed `START_URL` in your browser and confirm it loads normally.


In [1]:
import re
import time
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup


## 1) Settings and helper functions

In [2]:
BASE = "https://www.bbc.com"
START_URL = "https://www.bbc.com/business"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def get_soup(url: str) -> BeautifulSoup:
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return BeautifulSoup(r.text, "lxml")

def is_article_href(href: str) -> bool:
    if not href:
        return False
    href = href.split("?")[0]
    # Typical BBC article URL patterns (recent + older)
    return (
        href.startswith("/news/")
        and (
            "/articles/" in href
            or re.search(r"/news/business-\d+", href)  # older style
        )
    )


## 2) Collect article links from the Business landing page

In [3]:
soup = get_soup(START_URL)

urls = []
seen = set()

for a in soup.select("main a[href]"):
    href = a.get("href", "")
    if is_article_href(href):
        full = urljoin(BASE, href.split("?")[0])
        if full not in seen:
            seen.add(full)
            urls.append(full)

print("START_URL:", START_URL)
print("Article links found:", len(urls))
urls[:20]


START_URL: https://www.bbc.com/business
Article links found: 24


['https://www.bbc.com/news/articles/cge89x7re74o',
 'https://www.bbc.com/news/articles/cddn6yyr66yo',
 'https://www.bbc.com/news/articles/cwy8dxz1g7zo',
 'https://www.bbc.com/news/articles/c3ewje4xk3yo',
 'https://www.bbc.com/news/articles/c0ljpxek5w2o',
 'https://www.bbc.com/news/articles/cd6z05p56xyo',
 'https://www.bbc.com/news/articles/ce82xgd2g3yo',
 'https://www.bbc.com/news/articles/cx2y85xdyw3o',
 'https://www.bbc.com/news/articles/cddg885344do',
 'https://www.bbc.com/news/articles/cg7y45kxvp9o',
 'https://www.bbc.com/news/articles/cy57l501v2yo',
 'https://www.bbc.com/news/articles/cpv8nmr97rwo',
 'https://www.bbc.com/news/articles/cwy8dw5zk7yo',
 'https://www.bbc.com/news/articles/c78x9256pn7o',
 'https://www.bbc.com/news/articles/c0rjg7yny4zo',
 'https://www.bbc.com/news/articles/cr5lmg9l1y7o',
 'https://www.bbc.com/news/articles/cnv6j9vrn06o',
 'https://www.bbc.com/news/articles/c0q3dvww5pqo',
 'https://www.bbc.com/news/articles/czj17w3n1e8o',
 'https://www.bbc.com/news/arti

## 3) Visit each article and extract title, published time, and summary

In [4]:
rows = []

for i, url in enumerate(urls, start=1):
    try:
        s = get_soup(url)

        # Title (prefer OpenGraph)
        og_title = s.select_one("meta[property='og:title']")
        title = (og_title.get("content") or "").strip() if og_title else ""
        if not title:
            h1 = s.find("h1")
            title = h1.get_text(" ", strip=True) if h1 else ""

        # Summary (prefer OpenGraph/description)
        og_desc = s.select_one("meta[property='og:description']")
        summary = (og_desc.get("content") or "").strip() if og_desc else ""
        if not summary:
            meta_desc = s.select_one("meta[name='description']")
            summary = (meta_desc.get("content") or "").strip() if meta_desc else ""

        # Published time (if present)
        published = ""
        t = s.find("time")
        if t:
            published = (t.get("datetime") or t.get_text(" ", strip=True) or "").strip()

        rows.append({
            "url": url,
            "title": title,
            "published": published,
            "summary": summary,
        })

        if i % 10 == 0:
            print(f"processed {i}/{len(urls)}")

        time.sleep(0.3)  # be polite

    except Exception as e:
        rows.append({
            "url": url,
            "title": "",
            "published": "",
            "summary": "",
            "error": str(e),
        })

df = pd.DataFrame(rows, columns=["url", "title", "published", "summary"])
df.to_csv("bbc_business_articles.csv", index=False, encoding="utf-8")
print("Saved: bbc_business_articles.csv")
df.head(10)


processed 10/24
processed 20/24
Saved: bbc_business_articles.csv


,url,title,published,summary
0,https://www.bbc.com/news/articles/cge89x7re74o,Cuba oil refinery fire hits as fuel crisis dee...,2026-02-14T03:03:27.522Z,"The fire, which Cuban officials say was contai..."
1,https://www.bbc.com/news/articles/cddn6yyr66yo,Andrew facing claim he shared Treasury documen...,2026-02-14T02:03:37.936Z,Reports suggest the former prince shared a Tre...
2,https://www.bbc.com/news/articles/cwy8dxz1g7zo,Amazon's Ring ends deal with surveillance firm...,2026-02-13T21:09:19.058Z,A Super Bowl advert had sparked new scrutiny o...
3,https://www.bbc.com/news/articles/c3ewje4xk3yo,The US economy is growing - so where are all t...,2026-02-13T00:05:12.475Z,"As hiring rates and job openings drop, some wo..."
4,https://www.bbc.com/news/articles/c0ljpxek5w2o,Is eating out too expensive now? Families say ...,2026-02-14T01:26:44.769Z,The restaurant industry says it is facing a do...
5,https://www.bbc.com/news/articles/cd6z05p56xyo,Inflation eases in US as prices for used cars ...,2026-02-13T18:22:32.050Z,"Prices rose by 2.4% in the year to January, th..."
6,https://www.bbc.com/news/articles/ce82xgd2g3yo,Head of Dubai-based ports giant DP World quits...,2026-02-13T18:18:19.706Z,Sultan Ahmed bin Sulayem’s exit comes after fi...
7,https://www.bbc.com/news/articles/cx2y85xdyw3o,"The Dutch love four-day working weeks, but are...",2026-02-12T00:00:05.402Z,The Netherlands has the lowest working hours i...
8,https://www.bbc.com/news/articles/cddg885344do,The shadowy world of abandoned oil tankers,2026-02-09T00:09:30.116Z,A growing number of tankers and other commerci...
9,https://www.bbc.com/news/articles/cg7y45kxvp9o,Get a grip: Robotics firms struggle to develop...,2026-02-13T00:04:16.888Z,Developing a durable and affordable hand is on...


## Troubleshooting
- If `Article links found: 0`, print the page title and first 500 characters of HTML to see if you received a consent/blocked page.
- If many rows have empty `title`, BBC may be serving different templates; use the `<h1>` fallback already included.


In [5]:
# Optional: quick debug if no links were found
if len(urls) == 0:
    html = requests.get(START_URL, headers=HEADERS, timeout=30).text
    print("First 500 chars of HTML:\n", html[:500])
